In [12]:
import numpy as np
import pandas as pd

from scipy.stats import norm
import matplotlib.pyplot as plt

In [13]:
def simulated_asset_prices(S_0, r, T, sigma, no_paths= 100, antithetic= False):

  no_steps = 365
  dt = T/no_steps

  rng = np.random.default_rng(50)

  Z = rng.standard_normal((no_paths, no_steps))

  if no_paths > 1:
    Z = (Z - np.mean(Z, axis=0)) / np.std(Z, axis= 0)

  if antithetic:
    Z = np.vstack((Z, -Z))
    no_paths *= 2
  X = np.zeros([no_paths, no_steps + 1])
  time = np.zeros([no_steps + 1])

  X[:, 0] = np.log(S_0)

  for i in range(no_steps):
    X[:, i + 1] = X[:, i] + (r - 0.5 * np.square(sigma)) * dt + sigma * np.sqrt(dt) * Z[:, i]
    time[i + 1] = time[i] + dt

  S = np.exp(X)


  paths = {'time': time, 'S': S}

  return paths

In [14]:
def European_Option_pricing(S_0, r, T, sigma, K, type, no_paths= 100, antithetic= False):

  paths = simulated_asset_prices(S_0, r, T, sigma, no_paths, antithetic)
  S_T = paths['S'][:, -1]
  discount_factor = np.exp(-r * T)

  if type == "call":
    Euro_payoff = np.maximum(S_T - K, 0)

  elif type == "put":
    Euro_payoff = np.maximum(K - S_T, 0)

  else:
    raise ValueError("Option type should be either 'call' or 'put' ")

  price = discount_factor * np.mean(Euro_payoff)

  stdErr = np.exp(-r * T) * np.std(Euro_payoff, ddof= 1)/np.sqrt(no_paths)

  return price, stdErr

In [15]:
r = 0.045
K = 110
sigma = 0.35
T = 1
S_0 = 100

In [16]:
price, stdErr = European_Option_pricing(S_0, r, T, sigma, K, "call", no_paths= 10000, antithetic= True)
price

np.float64(11.580056480213676)

In [17]:
from scipy.stats import norm

def Black_Schole_price(S_0, r, T, K, sigma, type):
  d_1 = (np.log(S_0/K) + (r + (np.square(sigma) / 2)) * T) / (sigma * np.sqrt(T))
  d_2 = d_1 - sigma * np.sqrt(T)

  payoff = S_0 * norm.cdf(d_1) - np.exp(-r * T) * K * norm.cdf(d_2)

  return payoff

In [18]:
BS_price = Black_Schole_price(S_0, r, T, K, sigma, "call")
BS_price

np.float64(11.816007270287429)

In [19]:
left_CI_99 = price - 2.576 * stdErr
right_CI_99 = price + 2.576 * stdErr

print(f"The Black-Schole price {BS_price.round(5)} lies within the 99% confidence interval [{left_CI_99.round(5)}, {right_CI_99.round(5)}].")

The Black-Schole price 11.81601 lies within the 99% confidence interval [10.97817, 12.18194].
